# Dubai Real Estate Market Analysis (2020–2026)

Where should you actually buy in Dubai if you're investing, not just admiring the skyline?

Analysis of 50,000+ Dubai property sale transactions to identify pricing trends, 
top-performing communities, and rental yield opportunities for investors.

**Tools used:** Python (pandas), SQL (SQLite), Power BI

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/sergionefedov/dubai-real-estate-sales-and-rentals-20202026/secondary_sales.csv')
df.head()

## About This Data

This dataset combines real Dubai anchors like community locations, Dubai Metro 
stations, UAE Central Bank interest rate history, and base prices anchored to 
Dubai Land Department and Property Finder averages - with a hedonic pricing 
model to generate realistic individual listings. 

Individual listing prices are synthetically modeled rather than scraped real 
transactions (scraping real estate platforms violates their terms of service), 
but the underlying trends reflect real, documented Dubai market events: the 
COVID-era dip, the 2022 Golden Visa expansion, the 2023 peak, and the 
2024–2026 cooling.

[Source: Dubai Real Estate Sales & Rentals (2020-2026), Kaggle](https://www.kaggle.com/datasets/sergionefedov/dubai-real-estate-sales-and-rentals-20202026)

## Cleaning the Data

Dropping rows with missing price or area (can't analyze what isn't there), and 
extracting the year from each listing's date so I can look at trends over time.

In [7]:
df = df.dropna(subset=['price_usd', 'area_sqft'])
df['date_listed'] = pd.to_datetime(df['date_listed'])
df['year'] = df['date_listed'].dt.year

,id,date_listed,community,zone,is_freehold,lat,lon,property_category,property_type,bedrooms,...,metro_station,metro_line,metro_distance_min,metro_distance_type,to_burj_khalifa_km,price_usd,price_per_sqft_usd,price_per_m2_usd,mortgage_rate_at_listing,year
0,S000001,2024-11-23,DIFC,DIFC,True,25.21053,55.29453,apartment,4BR_penthouse,4,...,Financial Centre,Red,6,walk,2.51,4806100,1314,14141,6.15,2024
1,S000002,2023-07-05,DIFC,DIFC,True,25.21330,55.28617,apartment,3BR,3,...,Emirates Towers,Red,13,walk,2.15,2287400,1283,13813,6.90,2023
2,S000003,2022-01-22,Karama,Bur Dubai,False,25.24835,55.29501,apartment,1BR,1,...,BurJuman (G),Green,14,walk,6.05,167100,203,2184,1.90,2022
3,S000004,2021-10-28,The Valley,Dubai-Al Ain Road,True,24.95989,55.48870,villa,5BR_villa,5,...,Discovery Gardens,Red,151,drive,34.09,832000,154,1658,1.90,2021
4,S000005,2025-05-30,Sobha Hartland,MBR City,True,25.17393,55.28697,apartment,3BR,3,...,Financial Centre,Red,23,drive,2.88,1323400,660,7104,6.15,2025


In [8]:
# Question 1: Which community has the highest average price per sqft?
avg_price_by_area = df.groupby('community')['price_per_sqft_usd'].mean().sort_values(ascending=False)
print(avg_price_by_area.head(10))

community
Bulgari Resort             2276.821849
Jumeirah Bay Island        1592.004975
Pearl Jumeira              1127.813170
DIFC                       1066.754653
La Mer                     1013.036332
Emirates Hills              910.772652
Downtown Dubai              898.110016
Palm Jumeirah               885.709507
World Islands               859.608997
Madinat Jumeirah Living     844.143541
Name: price_per_sqft_usd, dtype: float64


In [9]:
# Question 2: How has price changed over time (2020-2026)?
yearly_avg = df.groupby('year')['price_per_sqft_usd'].mean()
print(yearly_avg)

year
2020    299.584503
2021    332.160814
2022    430.585565
2023    515.836925
2024    535.344397
2025    534.396182
2026    524.437669
Name: price_per_sqft_usd, dtype: float64


In [10]:
# Question 3: Which property type has the highest price per sqft?
avg_by_type = df.groupby('property_type')['price_per_sqft_usd'].mean().sort_values(ascending=False)
print(avg_by_type.head(10))

property_type
4BR_penthouse    604.757324
3BR              517.110402
2BR              479.528640
6BR_villa        457.392577
5BR_villa        436.980612
4BR_villa        423.292005
3BR_villa        416.171845
1BR              404.887341
studio           393.026937
Name: price_per_sqft_usd, dtype: float64


In [11]:
#Freehold vs non-freehold pricing
freehold_comparison = df.groupby('is_freehold')['price_per_sqft_usd'].mean()
print(freehold_comparison)

#Price by bedroom count
by_bedrooms = df.groupby('bedrooms')['price_per_sqft_usd'].mean().sort_values()
print(by_bedrooms)

is_freehold
False    240.032995
True     473.860059
Name: price_per_sqft_usd, dtype: float64
bedrooms
0    393.026937
1    404.887341
5    436.980612
6    457.392577
3    468.635350
4    472.458121
2    479.528640
Name: price_per_sqft_usd, dtype: float64


### Bonus: Best Value Areas for Family Homes (3+ Bedrooms)

In [ ]:
# If someone wants a family home (3+ bedrooms) on a budget, where should they look?
family_homes = df[df['bedrooms'] >= 3]
best_value_family = family_homes.groupby('community')['price_per_sqft_usd'].mean().sort_values().head(10)
print(best_value_family)

In [ ]:
# Does distance to metro correlate with price?
correlation = df['metro_distance_min'].corr(df['price_per_sqft_usd'])
print(f"Correlation between metro distance and price: {correlation:.3f}")

## Real SQL Queries

Everything above used pandas. This section writes actual SQL - including a JOIN 
across two tables

In [13]:
import sqlite3

# Loading metro stations data — this file has station names and metro lines
metro = pd.read_csv('/kaggle/input/datasets/sergionefedov/dubai-real-estate-sales-and-rentals-20202026/metro_stations.csv')

# Creating an in-memory SQL database and loading both tables into it
conn = sqlite3.connect(':memory:')
df.to_sql('sales', conn, index=False)
metro.to_sql('metro', conn, index=False)

# SQL JOIN: comparing average price by metro LINE (not just station)
query = '''
SELECT m.line, ROUND(AVG(s.price_per_sqft_usd), 0) as avg_price_per_sqft
FROM sales s
JOIN metro m ON s.metro_station = m.station_name
GROUP BY m.line
ORDER BY avg_price_per_sqft DESC
'''
result = pd.read_sql(query, conn)
print(result)

    line  avg_price_per_sqft
0    Red               455.0
1  Green               414.0


In [14]:
# Ranking properties by price within specific high-demand communities
query2 = '''
SELECT community, price_per_sqft_usd,
       RANK() OVER (PARTITION BY community ORDER BY price_per_sqft_usd DESC) as price_rank
FROM sales
WHERE community IN ('DIFC', 'Palm Jumeirah', 'Downtown Dubai')
'''
result2 = pd.read_sql(query2, conn)
print(result2.head(15))

   community  price_per_sqft_usd  price_rank
0       DIFC                2020           1
1       DIFC                1960           2
2       DIFC                1905           3
3       DIFC                1904           4
4       DIFC                1903           5
5       DIFC                1866           6
6       DIFC                1848           7
7       DIFC                1829           8
8       DIFC                1829           8
9       DIFC                1805          10
10      DIFC                1766          11
11      DIFC                1734          12
12      DIFC                1732          13
13      DIFC                1697          14
14      DIFC                1697          14


## The Real Question: Where's the Best Rental Yield?

Price alone doesn't tell you if somewhere is a good investment. Combining sales 
and rental data to calculate yield answers the question an actual investor 
would ask: where does buying actually pay off?

In [15]:
# Loading rentals data
rentals = pd.read_csv('/kaggle/input/datasets/sergionefedov/dubai-real-estate-sales-and-rentals-20202026/rentals.csv')

# Average purchase price per sqft by community (from existing sales data)
avg_sale = df.groupby('community')['price_per_sqft_usd'].mean().rename('avg_sale_price_per_sqft')

# Average annual rent per sqft by community
avg_rent = rentals.groupby('community')['rent_per_sqft_usd'].mean().rename('avg_annual_rent_per_sqft')

# Combining both into one table
combined = pd.concat([avg_sale, avg_rent], axis=1).dropna()

# Rental yield = annual rent / purchase price × 100
combined['rental_yield_pct'] = (combined['avg_annual_rent_per_sqft'] / combined['avg_sale_price_per_sqft']) * 100

combined = combined.sort_values('rental_yield_pct', ascending=False)

print("BEST RENTAL YIELD AREAS (best for investors):")
print(combined.head(10))
print()
print("LOWEST YIELD AREAS (prestige areas, weaker returns):")
print(combined.tail(5))

BEST RENTAL YIELD AREAS (best for investors):
                   avg_sale_price_per_sqft  avg_annual_rent_per_sqft  \
community                                                              
The Valley                      185.499093                 19.885522   
Dubai South                     125.879252                 12.948097   
Sustainable City                267.606504                 27.420195   
Green Community                 234.217600                 23.750831   
Damac Hills 2                   184.740558                 18.496528   
Expo City                       203.508503                 20.304183   
Tilal Al Ghaf                   312.163580                 30.736301   
The Meadows                     478.545736                 47.000000   
Arabian Ranches 3               460.881849                 44.616822   
The Springs                     415.496785                 39.754839   

                   rental_yield_pct  
community                            
The Valley   

## Key Findings

- **Bulgari Resort, Jumeirah Bay Island, and Pearl Jumeira** are Dubai's most 
  expensive communities by price per sqft
- Prices rose ~78% from 2020 to 2024, then flattened through 2025–2026 -
  matching real, documented market events (Golden Visa expansion, post-rally cooling)
- Properties near the **Red Line metro** carry a ~10% price premium over Green Line
- **The most "impressive" areas are often the worst investments:** Downtown Dubai 
  and luxury areas have the lowest rental yields (~6%), while The Valley and 
  Dubai South offer nearly double (~10-11%)

## Limitations

Individual listing prices are synthetically generated using a hedonic pricing 
model, not scraped real transactions - this is suitable for analytical 
methodology and market-pattern practice, not for valuing a specific real property.

Data source: [Dubai Real Estate Sales & Rentals (2020-2026), Kaggle](https://www.kaggle.com/datasets/sergionefedov/dubai-real-estate-sales-and-rentals-20202026)

df.to_csv('dubai_secondary_sales_cleaned.csv', index=False)